# 09 · spark-submit & Deploy Modes (Case C)

**Theory**: docs/01-from-mapreduce-to-spark.md ("Client vs. Cluster mode")

**Prerequisite**: `make up-hadoop` still running.

Every notebook so far has used **client mode** implicitly — the Driver is
this very process (or, for Cases B/D, the `spark-connect` container acting
as a permanent Driver). This lab makes the deploy-mode distinction explicit
by using `spark-submit` directly from a terminal, both ways.

## Client mode: the pattern you've been using all lab

Run this in a **terminal** (not in the notebook — `spark-submit` starts its
own JVM and blocks until the job finishes):

```bash
cat <<'EOF' > /tmp/word_count_job.py
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("word-count-client")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.driver.host", "host.docker.internal")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .getOrCreate()
)
df = spark.range(0, 10_000_000)
print(f"Count: {df.count()}")
spark.stop()
EOF

uv run spark-submit --master yarn --deploy-mode client /tmp/word_count_job.py
```

Notice the driver's logs print directly in **your terminal** — that's client
mode: your machine IS the Driver, watching the whole execution live.

## Cluster mode: driver runs inside YARN itself

```bash
uv run spark-submit --master yarn --deploy-mode cluster /tmp/word_count_job.py
```

This time your terminal only shows the **submission** succeeding — the
`print()` output does *not* appear locally, because the Driver process is
now running inside a YARN container (scheduled on nodemanager1 or
nodemanager2), not on your machine. To see its output, find the Application
ID printed by spark-submit and check its logs from the ResourceManager UI
(http://localhost:8088 -> the application -> Logs), or:

```bash
docker exec -it resourcemanager yarn logs -applicationId <application_id>
```

> 💡 **This is the deploy-mode trade-off from docs/01, made concrete**:
> cluster mode means your job survives your laptop disconnecting — because
> your laptop was never running the Driver in the first place. Client mode
> (which the shared `spark.driver.host=host.docker.internal` trick in
> `lab_utils.py` makes possible from your HOST) trades that resilience for
> the interactive, see-everything-live workflow this whole lab is built
> around.

## Watching it in the ResourceManager UI

For either run, open http://localhost:8088 while the job is in flight:

1. Click the Application ID -> see the Application Master's node and the
   number of containers (executors) allocated.
2. Note **where** the Application Master runs: in client mode it's a
   lightweight coordinator; in cluster mode, it *is* your Driver.
3. Compare the "Tracking UI" link — in cluster mode it points to a Spark UI
   running inside the YARN-managed Application Master itself.